In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q sentence-transformers chromadb rank_bm25 \
                langgraph langchain langchain-openai groq
!nvidia-smi | head -5

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Sat May  9 13:33:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |


In [ ]:
import json, shutil, os
from sentence_transformers import SentenceTransformer
import chromadb

# Chunks
chunks = [json.loads(l) for l in open("/content/drive/MyDrive/chunks.jsonl", encoding='utf-8')]
print(f"Chunks: {len(chunks)}")

# Embedding model
model = SentenceTransformer('AITeamVN/Vietnamese_Embedding', trust_remote_code=True, device='cuda')
print(f"Model loaded, dim={model.get_sentence_embedding_dimension()}")

# Copy chromadb từ Drive về local (FUSE chậm)
DRIVE_DB = "/content/drive/MyDrive/chromadb"
LOCAL_DB = "/content/chromadb"
if os.path.exists(LOCAL_DB): shutil.rmtree(LOCAL_DB)
shutil.copytree(DRIVE_DB, LOCAL_DB)
client = chromadb.PersistentClient(path=LOCAL_DB)
collection = client.get_or_create_collection(name="bctc_acbs", metadata={"hnsw:space": "cosine"})
print(f"Collection count: {collection.count()}")

Chunks: 6514


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

/tmp/ipykernel_24595/52064928.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded, dim={model.get_sentence_embedding_dimension()}")


Model loaded, dim=1024
Collection count: 6514


In [ ]:
import re
from rank_bm25 import BM25Okapi
from collections import defaultdict

VN_CHARS = 'a-z0-9àáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵđ'
token_re = re.compile(f'[{VN_CHARS}]+')
html_re = re.compile(r'<[^>]+>')

def tokenize(text):
    text = html_re.sub(' ', text).lower()
    return token_re.findall(text)

# BM25 index — thêm source vào corpus để bắt ticker
corpus_tokens = [tokenize(c['text'] + ' ' + c['source']) for c in chunks]
bm25 = BM25Okapi(corpus_tokens)
print(f"BM25 indexed: {len(corpus_tokens)} chunks")

# Map full name → ticker để query "Vinamilk" match được "VNM"
TICKER_NAMES = {
    'HPG': ['hòa phát', 'hoa phat'], 'VCB': ['vietcombank'],
    'VNM': ['vinamilk'], 'VHC': ['vĩnh hoàn', 'vinh hoan'],
    'VHM': ['vinhomes'], 'VIC': ['vingroup'], 'VRE': ['vincom retail'],
    'FPT': ['fpt'], 'BID': ['bidv'], 'CTG': ['vietinbank'],
    'TCB': ['techcombank'], 'MBB': ['mb bank', 'mbbank'],
    'STB': ['sacombank'], 'VPB': ['vpbank'], 'HDB': ['hdbank'],
    'VIB': ['vib'], 'TPB': ['tpbank'], 'LPB': ['lpbank'],
    'SSB': ['seabank'], 'ACB': ['acb'], 'BVH': ['bảo việt', 'bao viet'],
    'GAS': ['pv gas'], 'PLX': ['petrolimex'], 'POW': ['pv power'],
    'GVR': ['cao su'], 'DGC': ['đức giang'], 'MSN': ['masan'],
    'NVL': ['novaland'], 'PDR': ['phát đạt'], 'KDH': ['khang điền'],
    'SAB': ['sabeco'], 'BCM': ['becamex'],
}

def expand_query(q):
    q_low = q.lower()
    extra = [t for t, names in TICKER_NAMES.items()
             if any(n in q_low for n in names) and t.lower() not in q_low]
    return q + (' ' + ' '.join(extra) if extra else '')

def extract_ticker(query):
    q_low = query.lower()
    for ticker in TICKER_NAMES.keys():
        if re.search(rf'\b{ticker.lower()}\b', q_low):
            return ticker
    for ticker, names in TICKER_NAMES.items():
        if any(n in q_low for n in names):
            return ticker
    return None

def hybrid_search(query, top_k=3, k_rrf=60, candidates=20, source_boost=2.0):
    """BM25 + Dense + RRF fusion + ticker source boost"""
    expanded = expand_query(query)
    ticker = extract_ticker(query)

    bm25_scores = bm25.get_scores(tokenize(expanded))
    bm25_top = sorted(range(len(bm25_scores)), key=lambda i: -bm25_scores[i])[:candidates]

    q_emb = model.encode([f"query: {expanded}"], normalize_embeddings=True, convert_to_numpy=True)
    dense_res = collection.query(query_embeddings=q_emb.tolist(), n_results=candidates)
    dense_top = [int(id_.rsplit('_', 1)[-1]) for id_ in dense_res['ids'][0]]

    rrf = defaultdict(float)
    def add(rank, idx):
        s = 1.0 / (k_rrf + rank)
        if ticker and ticker.upper() in chunks[idx]['source'].upper():
            s *= source_boost
        rrf[idx] += s

    for rank, idx in enumerate(bm25_top): add(rank, idx)
    for rank, idx in enumerate(dense_top): add(rank, idx)

    return sorted(rrf.items(), key=lambda x: -x[1])[:top_k]

BM25 indexed: 6514 chunks


In [ ]:
import re, os
from google.colab import userdata
from groq import Groq


groq_client = Groq(api_key=userdata.get('GROQ_API_KEY'))
LLM_MODEL = "llama-3.3-70b-versatile"

def strip_html_for_llm(text):
  text = re.sub(r'</tr>\s*<tr[^>]*>', '\n', text)
  text = re.sub(r'</td>\s*<td[^>]*>', ' | ', text)
  text = re.sub(r'<[^>]+>', '', text)
  text = re.sub(r'[ \t]+', ' ', text)
  return text.strip()

def call_llm(prompt: str) -> str:
    """Wrapper cho Groq Llama (thay thế call_gemini cũ)"""
    resp = groq_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return resp.choices[0].message.content
def format_citation(source: str, page: int) -> str:
    """Convert source filename → human-readable citation per JD ACBS."""
    # BCTC: 3-letter ticker uppercase (HPG, TCB, VCB, FPT...)
    if len(source) <= 4 and source.isupper() and source.isalpha():
        return f"[BCTC {source} - Trang {page}]"

    # ACBS analysis: ACBS_<TICKER>_...
    if source.startswith("ACBS_"):
        parts = source.split("_")
        if len(parts) >= 2:
            return f"[Báo cáo ACBS {parts[1]} - Trang {page}]"

    # BSC: BSC_Morning-0505-..., BSC_Brief-0505-...
    if source.startswith("BSC_"):
        m = re.search(r'BSC_(\w+?)-(\d{2})(\d{2})-', source)
        if m:
            kind, dd, mm = m.group(1), m.group(2), m.group(3)
            return f"[BSC {kind} {dd}/{mm}/2026 - Trang {page}]"
        return f"[Báo cáo BSC - Trang {page}]"

    # VCSC Daily: VCSC_YYYYMMDD_DailyVN
    if source.startswith("VCSC_"):
        m = re.match(r'VCSC_(\d{4})(\d{2})(\d{2})_DailyVN', source)
        if m:
            yyyy, mm, dd = m.group(1), m.group(2), m.group(3)
            return f"[VCSC Daily {dd}/{mm}/{yyyy} - Trang {page}]"
        m2 = re.match(r'VCSC_(\w+)-', source)
        if m2:
            return f"[Báo cáo VCSC {m2.group(1).upper()} - Trang {page}]"
        return f"[Báo cáo VCSC - Trang {page}]"

    return f"[{source} - Trang {page}]"

# Test nhanh
for src, p in [("HPG", 5), ("TCB", 47), ("ACBS_HPG_Cap-nhat-nhanh-AGM-24042026", 2),
                ("BSC_Morning-0505-Viet-Nam-CPI-thang-4-546-yy", 1),
                ("VCSC_20260504_DailyVN", 8), ("VCSC_vhc-trien-vong-mang-ca-tra", 1)]:
    print(f"  {src:60s} → {format_citation(src, p)}")

  HPG                                                          → [BCTC HPG - Trang 5]
  TCB                                                          → [BCTC TCB - Trang 47]
  ACBS_HPG_Cap-nhat-nhanh-AGM-24042026                         → [Báo cáo ACBS HPG - Trang 2]
  BSC_Morning-0505-Viet-Nam-CPI-thang-4-546-yy                 → [BSC Morning 05/05/2026 - Trang 1]
  VCSC_20260504_DailyVN                                        → [VCSC Daily 04/05/2026 - Trang 8]
  VCSC_vhc-trien-vong-mang-ca-tra                              → [Báo cáo VCSC VHC - Trang 1]


In [ ]:
from langchain.tools import tool

@tool
def retrieve_document(query: str, top_k: int = 5) -> str:
      """Tìm thông tin tổng quát trong vector DB"""
      results = hybrid_search(query, top_k=top_k)
      parts = []
      for idx, _ in results:
          c = chunks[idx]
          readable = strip_html_for_llm(c['text'])[:1500]
          citation = format_citation(c['source'], c['page'])
          parts.append(f"{citation}\n{readable}")
      return "\n\n---\n\n".join(parts)


@tool
def extract_financial_metrics(company_ticker: str, metric: str) -> str:
    """Trích 1 KPI của 1 công ty từ BCTC"""
    query = f"{metric} của {company_ticker} Q1/2026"
    results = hybrid_search(query, top_k=5)
    relevant = [c for idx, _ in results
                for c in [chunks[idx]]
                if company_ticker.upper() in c['source'].upper()]
    if not relevant:
        return f"Không tìm thấy chunks của {company_ticker} cho '{metric}'"

    context = "\n\n".join(
        f"{format_citation(c['source'], c['page'])}\n{strip_html_for_llm(c['text'])[:1200]}"
        for c in relevant[:3]
    )
    prompt = f"""Trích CHÍNH XÁC giá trị "{metric}" của công ty {company_ticker} từ context.

CONTEXT:
{context}

Format trả lời (1 dòng):
{metric} của {company_ticker} = <giá trị> <đơn vị> <citation>

Trong đó <citation> COPY NGUYÊN VĂN từ context, vd "[BCTC TCB - Trang 47]" hoặc "[Báo cáo ACBS HPG - Trang 2]".

Nếu không tìm thấy: "Không tìm thấy {metric} của {company_ticker}"."""
    return call_llm(prompt)


@tool
def compare_companies(ticker_a: str, ticker_b: str, metric: str) -> str:
    """So sánh MỘT chỉ tiêu tài chính giữa HAI công ty.
    Dùng khi user hỏi 'So sánh X và Y về Z' hoặc 'X hay Y có Z lớn hơn'.
    Tự gọi extract_financial_metrics cho cả 2 cty rồi tổng hợp."""
    val_a = extract_financial_metrics.invoke({"company_ticker": ticker_a, "metric": metric})
    val_b = extract_financial_metrics.invoke({"company_ticker": ticker_b, "metric": metric})
    prompt = f"""Dựa vào 2 dữ liệu sau, so sánh "{metric}" giữa {ticker_a} và {ticker_b}.

Dữ liệu {ticker_a}: {val_a}
Dữ liệu {ticker_b}: {val_b}

Trả lời ngắn gọn (3-4 câu), nêu rõ:
- Cty nào có {metric} lớn hơn
- Chênh lệch tuyệt đối + % chênh lệch
- Giữ nguyên citation [Chunk N] từ dữ liệu gốc"""
    return call_llm(prompt)

/usr/local/lib/python3.12/dist-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

# Groq exposes OpenAI-compatible endpoint → tránh bug langchain-groq
llm = ChatOpenAI(
    model="openai/gpt-oss-120b",
    api_key=userdata.get('GROQ_API_KEY'),
    base_url="https://api.groq.com/openai/v1",
    temperature=0,
)

SYSTEM = """Bạn là chuyên gia phân tích tài chính của ACBS, hỗ trợ trả lời câu hỏi về 30 cổ phiếu VN30 và các báo cáo
phân tích ACBS/BSC/VCSC Q1/2026.

Quy tắc:
1. Phân loại câu hỏi để chọn tool đúng:
    - Câu hỏi tổng quát/định tính → retrieve_document
    - Hỏi 1 KPI cụ thể của 1 cty → extract_financial_metrics
    - So sánh 2 cty về 1 chỉ tiêu → compare_companies
2. Mỗi câu trả lời PHẢI có citation theo ĐÚNG format trong tool output, ví dụ:
    - [BCTC TCB - Trang 47]
    - [Báo cáo ACBS HPG - Trang 2]
    - [BSC Morning 05/05/2026 - Trang 1]
    - [VCSC Daily 04/05/2026 - Trang 8]
    COPY NGUYÊN VĂN citation từ context, KHÔNG đổi format.
3. Câu so sánh 2 cty: phải có 2 citation từ 2 BCTC khác nhau.
4. KHÔNG bịa số liệu. Nếu tool trả "Không tìm thấy", nói rõ với user.
5. Trả lời bằng tiếng Việt, ngắn gọn, chuyên nghiệp."""

tools = [retrieve_document, extract_financial_metrics, compare_companies]
agent = create_react_agent(llm, tools, prompt=SYSTEM)

print("Agent ready with 3 tools:")
for t in tools:
    print(f"  - {t.name}: {t.description[:80]}...")

Agent ready with 3 tools:
  - retrieve_document: Tìm thông tin tổng quát trong vector DB...
  - extract_financial_metrics: Trích 1 KPI của 1 công ty từ BCTC...
  - compare_companies: So sánh MỘT chỉ tiêu tài chính giữa HAI công ty.
    Dùng khi user hỏi 'So sánh ...


/tmp/ipykernel_24595/3484647565.py:31: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools, prompt=SYSTEM)


In [ ]:
TEST_QUERIES = [
    "ACBS đánh giá triển vọng dự án thép ray của Hòa Phát ra sao?",
    "Dự phòng rủi ro trái phiếu doanh nghiệp chưa niêm yết ngày 31/3/2026 của ngân hàng TMCP Kỹ thương Việt Nam Techcombank là bao nhiêu?",
    "So sánh tài sản ngắn hạn của HPG và FPT Q1/2026",
]

for i, q in enumerate(TEST_QUERIES, 1):
    print(f"\n{'='*78}\nQ{i}: {q}\n{'='*78}")
    result = agent.invoke({"messages": [("user", q)]})

    # Lấy danh sách tools đã gọi (để biết ReAct chọn tool nào)
    tools_used = []
    for msg in result['messages']:
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            tools_used.extend(tc['name'] for tc in msg.tool_calls)

    if tools_used:
        print(f"  🔧 Tool(s): {' → '.join(tools_used)}\n")

    # Chỉ in câu trả lời cuối cùng (AIMessage cuối cùng)
    final = result['messages'][-1].content
    print(f"  💬 Trả lời:\n{final}")


Q1: ACBS đánh giá triển vọng dự án thép ray của Hòa Phát ra sao?
  🔧 Tool(s): retrieve_document

  💬 Trả lời:
ACBS nhận định dự án thép ray của Hòa Phát có triển vọng tích cực. Theo báo cáo, nhà máy thép ray dự kiến sẽ cho ra sản phẩm đầu tiên vào Q2/2027 với công suất tổng khoảng 700.000 tấn (200.000 tấn thép ray + 500.000 tấn thép hình), đã có đơn đặt hàng cho dự án đường sắt cao tốc Hà Nội‑Quảng Ninh và dự kiến hoàn thành năm 2028. Dự án này được xem là động lực tăng trưởng chủ yếu cho mảng thép cốt lõi, góp phần đẩy mạnh doanh thu và lợi nhuận năm 2026, đồng thời ACBS đã nâng giá mục tiêu HPG lên 35.700 đồng, tăng 28 % so với giá hiện tại [Báo cáo ACBS HPG - Trang 2].

Q2: Dự phòng rủi ro trái phiếu doanh nghiệp chưa niêm yết ngày 31/3/2026 của ngân hàng TMCP Kỹ thương Việt Nam Techcombank là bao nhiêu?
  🔧 Tool(s): extract_financial_metrics

  💬 Trả lời:
Dự phòng rủi ro trái phiếu doanh nghiệp chưa niêm yết ngày 31/3/2026 của Ngân hàng TMCP Kỹ thương Việt Nam (Techcombank) là **4

In [ ]:
TEST_QUERIES = [
    "Quy trình ACBS phân tích cổ phiếu ngân hàng gồm những bước nào?",
    "Tiêu chuẩn khuyến nghị MUA của ACBS là gì?"
]

for i, q in enumerate(TEST_QUERIES, 1):
    print(f"\n{'='*78}\nQ{i}: {q}\n{'='*78}")
    result = agent.invoke({"messages": [("user", q)]})

    # Lấy danh sách tools đã gọi (để biết ReAct chọn tool nào)
    tools_used = []
    for msg in result['messages']:
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            tools_used.extend(tc['name'] for tc in msg.tool_calls)

    if tools_used:
        print(f"  🔧 Tool(s): {' → '.join(tools_used)}\n")

    # Chỉ in câu trả lời cuối cùng (AIMessage cuối cùng)
    final = result['messages'][-1].content
    print(f"  💬 Trả lời:\n{final}")


Q1: Quy trình ACBS phân tích cổ phiếu ngân hàng gồm những bước nào?
  🔧 Tool(s): retrieve_document

  💬 Trả lời:
Quy trình ACBS phân tích cổ phiếu ngân hàng gồm những bước sau:

1. Thu thập và phân tích dữ liệu về cổ phiếu ngân hàng, bao gồm cả dữ liệu tài chính và phi tài chính.
2. Đánh giá và phân tích các yếu tố ảnh hưởng đến giá cổ phiếu, bao gồm cả yếu tố kinh tế vĩ mô và yếu tố cụ thể của công ty.
3. Xác định mục tiêu giá và suất sinh lợi cổ tức cho cổ phiếu.
4. So sánh giá mục tiêu với giá thị trường để đưa ra khuyến nghị mua, bán hoặc trung lập.
5. Cập nhật và điều chỉnh khuyến nghị dựa trên sự thay đổi của thị trường và các yếu tố ảnh hưởng đến giá cổ phiếu.

[Báo cáo ACBS HPG - Trang 2]
[Báo cáo ACBS MSN - Trang 4]
[Báo cáo ACBS DPM - Trang 2]
[Báo cáo ACBS DPM - Trang 3]
[Báo cáo ACBS DPM - Trang 1]

Q2: Tiêu chuẩn khuyến nghị MUA của ACBS là gì?
  🔧 Tool(s): retrieve_document → retrieve_document

  💬 Trả lời:
Tiêu chuẩn khuyến nghị MUA của ACBS là nếu giá mục tiêu (bao gồm

In [ ]:
TEST_QUERIES = [
    "Template báo cáo cập nhật nhanh ACBS có những mục nào?",
]

for i, q in enumerate(TEST_QUERIES, 1):
    print(f"\n{'='*78}\nQ{i}: {q}\n{'='*78}")
    result = agent.invoke({"messages": [("user", q)]})

    # Lấy danh sách tools đã gọi (để biết ReAct chọn tool nào)
    tools_used = []
    for msg in result['messages']:
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            tools_used.extend(tc['name'] for tc in msg.tool_calls)

    if tools_used:
        print(f"  🔧 Tool(s): {' → '.join(tools_used)}\n")

    # Chỉ in câu trả lời cuối cùng (AIMessage cuối cùng)
    final = result['messages'][-1].content
    print(f"  💬 Trả lời:\n{final}")


Q1: Template báo cáo cập nhật nhanh ACBS có những mục nào?
  🔧 Tool(s): retrieve_document → retrieve_document → retrieve_document

  💬 Trả lời:
Nội dung báo cáo cập nhật nhanh ACBS bao gồm các mục như: Tóm tắt, Khuyến nghị, Giá mục tiêu, Thông tin cập nhật, Triển vọng, Điểm nhấn, Điểm lo ngại, v.v. [Báo cáo ACBS DPM - Trang 1]


In [ ]:
!pip install -q fastapi uvicorn pyngrok nest-asyncio

import re, threading, time
import nest_asyncio
import uvicorn
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from pyngrok import ngrok
from google.colab import userdata

nest_asyncio.apply()

# ============== FastAPI app ==============
app_api = FastAPI(title="ACBS RAG Agent", version="0.1")

app_api.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)


class ChatRequest(BaseModel):
    query: str


class ChatResponse(BaseModel):
    answer: str
    citations: list
    tools_used: list


@app_api.get("/health")
def health():
    return {
        "status": "ok",
        "chunks": len(chunks),
        "collection_count": collection.count(),
    }


CITATION_RE = re.compile(
    r'\[(?:BCTC [^\]]+|Báo cáo ACBS [^\]]+|BSC \w+ \d+/\d+/\d+ - Trang \d+|'
    r'VCSC Daily \d+/\d+/\d+ - Trang \d+|Báo cáo VCSC [^\]]+|'
    r'Thông tư TT-\d+[^\]]*)\]'
)


@app_api.post("/chat", response_model=ChatResponse)
def chat(req: ChatRequest):
    result = agent.invoke({"messages": [("user", req.query)]})
    tools_used = [
        tc['name']
        for msg in result['messages']
        if hasattr(msg, 'tool_calls') and msg.tool_calls
        for tc in msg.tool_calls
    ]
    final_answer = result['messages'][-1].content
    citations = sorted(set(m.group(0) for m in CITATION_RE.finditer(final_answer)))
    return ChatResponse(
        answer=final_answer,
        citations=citations,
        tools_used=tools_used,
    )


# ============== Setup ngrok + run uvicorn in background thread ==============
ngrok.set_auth_token(userdata.get('NGROK_AUTH_TOKEN'))
ngrok.kill()  # clear old tunnels

tunnel = ngrok.connect(8000)
PUBLIC_URL = tunnel.public_url   # ← attribute đúng

# Chạy uvicorn trong background thread (cell không block)
def _run_server():
    uvicorn.run(app_api, host="0.0.0.0", port=8000, log_level="warning")

server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()
time.sleep(3)  # đợi server boot

print("\n" + "="*70)
print(f"🌐 Public URL: {PUBLIC_URL}")
print("="*70)
print(f"\nTest từ PowerShell trên máy local:\n")
print(f'  curl {PUBLIC_URL}/health\n')
print(f'  curl -X POST {PUBLIC_URL}/chat `')
print(f'    -H "Content-Type: application/json" `')
print(f'    -d "{{\\"query\\": \\"Vốn chủ sở hữu của HPG là bao nhiêu?\\"}}"')
print("\n" + "="*70)
print("Server chạy BACKGROUND, cell không block. Tunnel sống đến khi đóng Colab.")


🌐 Public URL: https://countable-skied-turban.ngrok-free.dev

Test từ PowerShell trên máy local:

  curl https://countable-skied-turban.ngrok-free.dev/health

  curl -X POST https://countable-skied-turban.ngrok-free.dev/chat `
    -H "Content-Type: application/json" `
    -d "{\"query\": \"Vốn chủ sở hữu của HPG là bao nhiêu?\"}"

Server chạy BACKGROUND, cell không block. Tunnel sống đến khi đóng Colab.
